In [ ]:
import json
import os
import time
import numpy as np
import pandas as pd
import torch

from tabicl import TabICLForecaster

start_time = time.time()

def root_mean_squared_error(y_true, y_pred):
    y_true = np.asarray(y_true)
    y_pred = np.asarray(y_pred)
    return float(np.sqrt(np.mean((y_true - y_pred) ** 2)))

# ============================================================
# CONFIG
# ============================================================
DAYS_JSON = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\dataset_days.json"
DATA_DIR  = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\DataCleaning\clean"
OUT_DIR   = r"C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs"

countries = ["Denmark"]
#countries = ["Germany", "Ireland", "Portugal"]

days = ["day1"]

features = [
    "temperature_2m",
    "relative_humidity_2m",
    "wind_speed_10m",
    "precipitation",
    "direct_radiation",
    "price_eur_kwh",
]

PREDICTION_LENGTH = 96
CONTEXT_LENGTH = 672

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)

# ============================================================
# LOAD SPLIT DAYS
# ============================================================
with open(DAYS_JSON, "r") as f:
    dataset_days = json.load(f)

# ============================================================
# LOAD TABICL FORECASTER
# ============================================================
forecaster = TabICLForecaster(
    max_context_length=CONTEXT_LENGTH,
    temporal_features=None,       # repo default: index + datetime + periodic
    point_estimate="mean",        # use "median" if you want a more robust point forecast
    tabicl_config={
        "device": DEVICE,
        "verbose": False,
        # Optional:
        # "n_estimators": 8,
        # "batch_size": 8,
        # "kv_cache": False,
    },
)

# ============================================================
# HELPER FUNCTIONS
# ============================================================
def make_tabicl_context(series, item_id):
    """
    Convert one household series into TabICLForecaster format.

    Expected columns:
    - item_id
    - timestamp
    - target
    """
    context_df = series.reset_index()
    context_df.columns = ["timestamp", "target"]
    context_df["item_id"] = item_id
    return context_df[["item_id", "timestamp", "target"]]


def extract_tabicl_prediction(pred_df, item_id, prediction_length):
    """
    Robustly extract predictions from TabICLForecaster output.
    Different versions may name the prediction column differently.
    """
    pred_df = pred_df.reset_index()

    if "item_id" in pred_df.columns:
        pred_df = pred_df[pred_df["item_id"] == item_id]

    possible_cols = ["mean", "median", "target", "prediction", "pred"]

    pred_col = None
    for col in possible_cols:
        if col in pred_df.columns:
            pred_col = col
            break

    if pred_col is None:
        numeric_cols = pred_df.select_dtypes(include=[np.number]).columns.tolist()
        if len(numeric_cols) == 0:
            raise ValueError(f"Cannot find prediction column. Columns: {pred_df.columns.tolist()}")
        pred_col = numeric_cols[-1]

    y_pred = pred_df[pred_col].to_numpy()[:prediction_length]

    if len(y_pred) != prediction_length:
        raise ValueError(f"Expected {prediction_length} predictions, got {len(y_pred)}")

    return y_pred


# ============================================================
# MAIN LOOP
# ============================================================
# ============================================================
# MAIN LOOP
# ============================================================
rmse_results = []

for country in countries:

    country_start_time = time.time()

    print("Processing country:", country)

    data_path = rf"{DATA_DIR}\dataset_{country.capitalize()}.csv"
    df = pd.read_csv(data_path, index_col="timestamp", parse_dates=True).sort_index()

    households = [col for col in df.columns if col not in features]

    for day in days:
        print("   Day:", day)

        cutoff = pd.to_datetime(dataset_days[country][day])

        predictions_df_all_households = None
        rmse_households = []

        for household in households:

            household_start = time.time()

            s_train = df.loc[df.index < cutoff, household].dropna()

            if len(s_train) < CONTEXT_LENGTH:
                print(f"      Skipping {household}: not enough context ({len(s_train)} < {CONTEXT_LENGTH})")
                continue

            s_context = s_train.iloc[-CONTEXT_LENGTH:]

            y_true_series = df.loc[df.index >= cutoff, household].head(PREDICTION_LENGTH)

            if len(y_true_series) < PREDICTION_LENGTH:
                print(f"      Skipping {household}: not enough future observations")
                continue

            if y_true_series.isna().any():
                print(f"      Skipping {household}: NaNs in future truth")
                continue

            context_df = make_tabicl_context(s_context, item_id=household)

            try:
                pred_df = forecaster.predict_df(
                    context_df,
                    prediction_length=PREDICTION_LENGTH,
                )

                y_pred = extract_tabicl_prediction(
                    pred_df=pred_df,
                    item_id=household,
                    prediction_length=PREDICTION_LENGTH,
                )

            except Exception as e:
                print(f"      Skipping {household}: TabICL prediction failed: {e}")
                continue

            if predictions_df_all_households is None:
                predictions_df_all_households = pd.DataFrame(index=y_true_series.index)

            predictions_df_all_households[household] = y_pred

            y_true = y_true_series.to_numpy()

            rmse = root_mean_squared_error(y_true, y_pred)
            rmse_households.append(rmse)

            household_end = time.time()

            print(
                f"      {household} runtime: "
                f"{household_end - household_start:.2f} sec"
            )

        if len(rmse_households) == 0:
            print(f"      No valid households for {country} - {day}")
            continue

        avg_rmse_households = float(np.mean(rmse_households))

        rmse_results.append({
            "country": country,
            "day": day,
            "rmse": avg_rmse_households,
        })

        output = rf"{OUT_DIR}\TabICL_pred_{day}_{country.capitalize()}.csv"

        os.makedirs(os.path.dirname(output), exist_ok=True)

        predictions_df_all_households.to_csv(output, index=True)

        print("      Saved:", output)

    # ========================================================
    # COUNTRY RUNTIME
    # ========================================================
    country_runtime = time.time() - country_start_time

    print(
        f"\nTotal runtime for {country}: "
        f"{country_runtime:.2f} seconds"
    )

    # ========================================================
    # SAVE / UPDATE JSON
    # ========================================================
    json_path = os.path.join(
        OUT_DIR,
        f"time_spend_{country}.json"
    )

    if os.path.exists(json_path):
        with open(json_path, "r") as f:
            runtime_dict = json.load(f)
    else:
        runtime_dict = {}

    if "Foundational" not in runtime_dict:
        runtime_dict["Foundational"] = {}

    runtime_dict["Foundational"]["TabICL"] = country_runtime

    with open(json_path, "w") as f:
        json.dump(runtime_dict, f, indent=4)

    print(f"Saved/updated runtime JSON: {json_path}")

# ============================================================
# SUMMARY
# ============================================================
rmse_df = pd.DataFrame(rmse_results)

print("\nPer-day RMSE:")
print(rmse_df)

print("\nCross-validated RMSE per country:")
print(rmse_df.groupby("country")["rmse"].mean())

end_time = time.time()
print(f"Total runtime: {end_time - start_time:.2f} seconds")

Using device: cuda
Processing country: Denmark
   Day: day1


GPU 0:: 100%|██████████| 1/1 [00:01<00:00,  1.12s/it]


      home_1 runtime: 1.15 sec


GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.46it/s]


      home_2 runtime: 0.73 sec


...

      home_7 runtime: 0.71 sec


GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.23it/s]


      home_8 runtime: 0.85 sec


GPU 0:: 100%|██████████| 1/1 [00:00<00:00,  1.25it/s]

      home_9 runtime: 0.84 sec
      Saved: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\TabICL_pred_day1_Denmark.csv

Total runtime for Denmark: 7.63 seconds
Saved runtime JSON: C:\Users\CR58XM\Documents\GitHub\AAU_learning_to_predict_together_or_alone\Outputs\time_spend_Denmark.json

Per-day RMSE:
   country   day      rmse
0  Denmark  day1  1.385062

Cross-validated RMSE per country:
country
Denmark    1.385062
Name: rmse, dtype: float64
Total runtime: 7.63 seconds
